# 🚀 Semiconductor Image Restoration (NAFNet-SR) - Google Colab Training Notebook

This notebook trains the **NAFNet-SR** model on a GPU (T4 / V100 / A100) in Google Colab using branch `Kunal`.

### **Hyperparameters & Specs**
- **Architecture**: NAFNet-SR (`width=64`, ~9.6M parameters)
- **Loss Function**: Metrology Composite Loss (Charbonnier + Sobel Edge + 2D FFT + SSIM)
- **Patch Size**: `64x64` random crops -> `128x128` GT
- **Learning Rate**: `3e-4` (Cosine Annealing)
- **Batch Size**: `16` (or `32` on A100)
- **Epochs**: `200`


## Step 1: Check GPU Acceleration


In [ ]:
!nvidia-smi


## Step 2: Clone Repository (Branch `Kunal`)


In [ ]:
!git clone -b Kunal https://github.com/KJ-CORE/semicon_2026.git
%cd semicon_2026


## Step 3: Install Dependencies


In [ ]:
!pip install -r requirements.txt


## Step 4: Extract Dataset & Create Validation Split
*Upload `train.zip` and `Test_NoisyLR.zip` into `/content/semicon_2026/` before running this cell.*


In [ ]:
import os, zipfile, glob, shutil, random

# Extract train.zip if present
if os.path.exists('train.zip'):
    print('Extracting train.zip...')
    os.makedirs('data', exist_ok=True)
    with zipfile.ZipFile('train.zip', 'r') as z:
        z.extractall('data')
    print('Extraction complete!')

# Extract Test_NoisyLR.zip if present
if os.path.exists('Test_NoisyLR.zip'):
    print('Extracting Test_NoisyLR.zip...')
    os.makedirs('data/test', exist_ok=True)
    with zipfile.ZipFile('Test_NoisyLR.zip', 'r') as z:
        z.extractall('data/test')
    print('Test set extraction complete!')

# Create Train / Val split (10% validation)
if os.path.exists('data/train/NoisyLR') and not os.path.exists('data/val'):
    random.seed(42)
    os.makedirs('data/val/NoisyLR', exist_ok=True)
    os.makedirs('data/val/GT', exist_ok=True)
    files = sorted(glob.glob('data/train/NoisyLR/*.npy'))
    val_files = random.sample(files, k=int(len(files) * 0.1))
    for f in val_files:
        fname = os.path.basename(f)
        shutil.move(f, os.path.join('data/val/NoisyLR', fname))
        shutil.move(os.path.join('data/train/GT', fname), os.path.join('data/val/GT', fname))
    print(f'Created Val Split: {len(val_files)} samples moved to data/val/')


## Step 5: Start Model Training


In [ ]:
!python train.py --epochs 200 --batch_size 16 --lr 3e-4 --scale 2 --patch_size 64


## Step 6: Backup Checkpoint to Google Drive (Optional)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp weights/best_model.pt /content/drive/MyDrive/best_model.pt
print('Best model checkpoint saved to Google Drive!')


## Step 7: Run Evaluation / Inference


In [ ]:
!python eval.py --input_dir data/test/NoisyLR --output_dir data/output_restored --weights weights/best_model.pt --scale 2
